# 4A

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Load the OmicsCNGene.csv file and modify the column headers to have just the gene names
# OmicsCNGene.csv from Depmap
omics_data = pd.read_csv('data/OmicsCNGene.csv', index_col=0)
omics_data.columns = omics_data.columns.str.extract(r'([^\s]+)')[0]  # Extract gene names from column headers

# Step 2: Load the CDK11B dependency score data
cdk11b_dependency = pd.read_csv('../data/cn-dependency/CDK11B_CRISPR_Project_Score_CERES.csv')
cdk11b_dependency = cdk11b_dependency[['Depmap ID', 'CRISPR (Project Score, CERES)']]
cdk11b_dependency.set_index('Depmap ID', inplace=True)

# Step 3: Align the datasets using common cell lines
common_cell_lines = omics_data.index.intersection(cdk11b_dependency.index)
omics_data_aligned = omics_data.loc[common_cell_lines]
cdk11b_dependency_aligned = cdk11b_dependency.loc[common_cell_lines]

# Step 4: Correlate the CDK11B dependency scores with each column in the Omics CN data
correlation_results = omics_data_aligned.apply(
    lambda x: np.corrcoef(cdk11b_dependency_aligned['CRISPR (Project Score, CERES)'], x)[0, 1]
)

# Step 5: Sort the correlation results and prepare for plotting
correlation_sorted = correlation_results.sort_values(ascending=True)

# Step 6: Read in genes to highlight from genes.txt and remove duplicates
highlight_genes = pd.read_csv('../data/Genes.txt', header=None)[0].tolist()
highlight_genes = list(set(highlight_genes))  # Remove duplicates

# Step 7: Create a scatter plot with specified genes highlighted
plt.figure(figsize=(90, 60), dpi=300)

# Determine colors, highlighting any genes found in the genes.txt file
colors = ['red' if gene in highlight_genes else 'black' for gene in correlation_sorted.index]

# Plot all points with a single scatter command
plt.scatter(range(len(correlation_sorted))[::-1], correlation_sorted, color=colors, s=1500)  # Adjusted point size

# Customize plot spines
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(10)   # Left spine (y-axis)
ax.spines['bottom'].set_linewidth(10) # Bottom spine (x-axis)

# Add labels and title
plt.ylabel('CDK11B Dependency - \nCN Correlation', fontsize=220, fontweight='bold')
plt.xlabel('Gene Rank', fontsize=220, labelpad=60, fontweight='bold')
plt.title('Genome-wide Biomarker Analysis', fontsize=250, pad=600, fontweight='bold')

# Add arrows and labels for CDK11A, CDK11B, and CCNL2
genes = ['CDK11A','CCNL2', 'CDK11B']
y_offsets = [-0.1, -0.001, 0.04]  # Different vertical offsets for each gene

for gene, offset in zip(genes, y_offsets):
    if gene in correlation_sorted.index:
        x = len(correlation_sorted) - correlation_sorted.index.get_loc(gene) - 1
        y = correlation_sorted[gene]
    
        plt.annotate(gene,
                     xy=(x, y), 
                     xytext=(x + 6500, y + offset),  # Using different offset for each gene
                     arrowprops=dict(facecolor='blue', arrowstyle="->", lw=6),
                     fontsize=180, color='blue', ha='right')

# Add a legend with custom labels for the groups
plt.scatter([], [], color='red', label='Located on \nChr.1p36', s=1200)
plt.scatter([], [], color='black', label='Not located on \nChr.1p36', s=1200)
plt.legend(fontsize=240, frameon=False, loc='upper right',markerscale=3)
plt.yticks(fontsize=180)
#plt.xticks(fontsize=180)
# Save the plot
# plt.savefig('../Waterfall.svg', bbox_inches='tight', dpi=300, pad_inches=0.5)
plt.show()

# 4B

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load the CSV files
df1 = pd.read_csv('../data/crispr/1p36_CN.csv')  # Using 1p36_CN.csv for CN data
df2 = pd.read_csv('../data/crispr/CDK11B_CRISPR_Project_Score_CERES.csv')  # CRISPR scores

# Calculate the average across CDK11A, CDK11B, and CCNL2 columns
df1['average_CDK11_CCNL2'] = df1[['CDK11A', 'CDK11B', 'CCNL2']].mean(axis=1)

# Define percentile thresholds for categorizing copy number
deep_del_threshold = df1['average_CDK11_CCNL2'].quantile(0.1)
shallow_del_threshold = df1['average_CDK11_CCNL2'].quantile(0.4)

# Categorize copy number into "Deep Deletion," "Shallow Deletion," and "Copy Neutral / Gains"
df1['CN_category'] = pd.cut(df1['average_CDK11_CCNL2'], 
                            bins=[-float('inf'), deep_del_threshold, shallow_del_threshold, float('inf')], 
                            labels=['Deep Deletion', 'Shallow Deletion', 'Copy Neutral / Gain'])

# Merge the dataframes on depmap_id
merged_df = pd.merge(df1[['depmap_id', 'CN_category']], df2, left_on='depmap_id', right_on='Depmap ID')

# Set the context to increase font sizes
sns.set_context("talk", font_scale=7)

# Initialize the figure
plt.figure(figsize=(32,35), dpi=300)

custom_order = ['Copy Neutral / Gain', 'Shallow Deletion', 'Deep Deletion']

# Create violin plot
ax = sns.violinplot(x='CN_category', y='CRISPR (Project Score, CERES)', data=merged_df,
                    order=custom_order, color=".8", density_norm="count", inner='point', bw_adjust=1.5)

# Overlay scatter plot
sns.swarmplot(x='CN_category', y='CRISPR (Project Score, CERES)', data=merged_df,
              order=custom_order, color='black', size=25, ax=ax)

# Set x-ticks and labels
plt.xticks([0, 1, 2], ['Copy \n Neutral \n/ Gain', 'Shallow \nDeletion', 'Deep \nDeletion'], 
           fontsize=140, horizontalalignment='center')
ax.tick_params(axis='x', length=0, pad=15) 

ax.tick_params(axis='y', labelsize=120)
for tick in ax.get_yticklabels():
    tick.set_fontweight('bold')

# Customize plot spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(10)   # Left spine (y-axis)
ax.spines['bottom'].set_linewidth(10) # Bottom spine (x-axis)

# Calculate and plot the medians
means = merged_df.groupby('CN_category')['CRISPR (Project Score, CERES)'].mean().reindex(custom_order).values
for i, mean in enumerate(means):
    plt.plot(
        [i - 0.2, i + 0.2],  # Adjust the horizontal span to align with the violin width
        [mean, mean],    # Keep the y-coordinate constant at the median value
        color='red', 
        linestyle='-', 
        linewidth=12, zorder=10
    )

# Perform t-tests
wild_vs_shallow = ttest_ind(
    merged_df[merged_df['CN_category'] == 'Copy Neutral / Gain']['CRISPR (Project Score, CERES)'].dropna(),
    merged_df[merged_df['CN_category'] == 'Shallow Deletion']['CRISPR (Project Score, CERES)'].dropna(), alternative='two-sided', equal_var=False
)
shallow_vs_deep = ttest_ind(
    merged_df[merged_df['CN_category'] == 'Shallow Deletion']['CRISPR (Project Score, CERES)'].dropna(),
    merged_df[merged_df['CN_category'] == 'Deep Deletion']['CRISPR (Project Score, CERES)'].dropna(), alternative='two-sided', equal_var=False
)
wild_vs_deep = ttest_ind(
    merged_df[merged_df['CN_category'] == 'Copy Neutral / Gain']['CRISPR (Project Score, CERES)'].dropna(),
    merged_df[merged_df['CN_category'] == 'Deep Deletion']['CRISPR (Project Score, CERES)'].dropna(), alternative='two-sided', equal_var=False
)

# Add significance annotations
y_max = merged_df['CRISPR (Project Score, CERES)'].max()
#delta_high_medium = means[0] - means[1]
#delta_medium_low = means[1] - means[2]
#delta_high_low = means[0] - means[2]

def add_stat_annotation(ax, p_value, x1, x2, y, h, star_offset=-0.2, delta_offset=-0.1, color='k'):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=6.5, color=color)
    if p_value < 0.001:
        star_label = '***'
    elif p_value < 0.01:
        star_label = '**'
    elif p_value < 0.05:
        star_label = '*'
    else:
        star_label = 'ns'
    ax.text((x1 + x2) * 0.5, y + h + star_offset, star_label, ha='center', va='bottom', color=color, fontsize=240)
    #ax.text((x1 + x2) * 0.5, y + 0.5*h + delta_offset, f"Δ={delta:.2f}", ha='center', va='bottom', color='black', fontsize=90)

# Add significance bars, stars, and deltas
add_stat_annotation(plt.gca(), wild_vs_shallow.pvalue,  0, 1, y_max + 0.4, 0.1)
add_stat_annotation(plt.gca(), shallow_vs_deep.pvalue, 1, 2, y_max + 0.7, 0.1)
add_stat_annotation(plt.gca(), wild_vs_deep.pvalue,  0, 2, y_max + 1, 0.1)

# Add titles and labels
plt.title('CDK11: CRISPR Dependency', fontsize=180, pad=90, fontweight='bold')
plt.xlabel('Chr. 1p36 Copy Number', fontsize=180, labelpad=60, fontweight='bold')
plt.ylabel('CDK11B Dependency \nScore', fontsize=180, fontweight='bold')

# Save the plot
# plt.savefig('../CDK11B_CRISPR.svg', bbox_inches='tight', dpi=300, pad_inches=0.5)
plt.show()


In [ ]:
print(wild_vs_shallow.pvalue)
print(shallow_vs_deep.pvalue)
print(wild_vs_deep.pvalue)

# 4C

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Load omics gene expression data and clean column headers to get just gene names
# from depmap
omics_data = pd.read_csv(
    '../data/OmicsExpressionProteinCodingGenesTPMLogp1BatchCorrected.csv',
    index_col=0
)
omics_data.columns = omics_data.columns.str.extract(r'([^\s]+)')[0]  # Extract gene names

# Step 2: Load CDK11B dependency score data
cdk11b_dependency = pd.read_csv(
    '../data/crispr/CDK11B_CRISPR_Project_Score_CERES.csv'
)[['Depmap ID', 'CRISPR (Project Score, CERES)']].set_index('Depmap ID')

# Step 3: Align both datasets using common DepMap IDs (cell lines)
common_cell_lines = omics_data.index.intersection(cdk11b_dependency.index)
omics_data_aligned = omics_data.loc[common_cell_lines]
cdk11b_dependency_aligned = cdk11b_dependency.loc[common_cell_lines]

# Step 4: Compute correlation between CDK11B dependency and each gene's expression
correlation_results = omics_data_aligned.apply(
    lambda x: np.corrcoef(cdk11b_dependency_aligned['CRISPR (Project Score, CERES)'], x)[0, 1]
)

# Step 5: Sort correlations for waterfall plotting
correlation_sorted = correlation_results.sort_values(ascending=True)

# Step 6: Load list of genes to highlight
highlight_genes = pd.read_csv('../data/Genes.txt', header=None)[0].unique().tolist()

# Step 7: Plot
plt.figure(figsize=(90, 60), dpi=300)
x_vals = list(range(len(correlation_sorted)))[::-1]

# Identify red (highlighted) and black (non-highlighted) genes
red_indices = [i for i, gene in enumerate(correlation_sorted.index) if gene in highlight_genes]
black_indices = [i for i, gene in enumerate(correlation_sorted.index) if gene not in highlight_genes]

# Plot black points first
# Plot black dots underneath
plt.scatter(
    [x_vals[i] for i in black_indices],
    correlation_sorted.iloc[black_indices],
    color='black', s=1500, zorder=1  # reduce opacity
)

# Plot red dots on top
plt.scatter(
    [x_vals[i] for i in red_indices],
    correlation_sorted.iloc[red_indices],
    color='red', s=500, zorder=2  # bold white outline
)


# Customize axes
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(10)
ax.spines['bottom'].set_linewidth(10)

plt.ylabel('CDK11B Dependency - \nGene Expression Correlation', fontsize=220, fontweight='bold', labelpad=100)
plt.xlabel('Gene Rank', fontsize=220, fontweight='bold', labelpad=60)
plt.title('Genome-wide \nGene Expression Analysis', fontsize=250, fontweight='bold', pad=600)
plt.yticks(fontsize=180)

# Annotate specific genes with arrows
genes_to_annotate = ['CDK11A', 'CCNL2', 'CDK11B']
y_offsets = [-0.1, -0.001, 0.04]

for gene, offset in zip(genes_to_annotate, y_offsets):
    if gene in correlation_sorted.index:
        x = len(correlation_sorted) - correlation_sorted.index.get_loc(gene) - 1
        y = correlation_sorted[gene]
        plt.annotate(
            gene,
            xy=(x, y),
            xytext=(x + 4000, y + offset),
            arrowprops=dict(facecolor='blue', arrowstyle="->", lw=6),
            fontsize=180, color='blue', ha='right'
        )

# Add legend for red vs. black dots
plt.scatter([], [], color='red', label='Located on \nChr. 1p36', s=1200)
plt.scatter([], [], color='black', label='Not located on \nChr. 1p36', s=1200)
plt.legend(fontsize=220, frameon=False, loc='upper right', markerscale=3)

# Save or show the plot
# plt.savefig('../Waterfall_Gene_Exp.png', bbox_inches='tight', dpi=300, pad_inches=0.5)
plt.show()


# 4D

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load the CSV files
df1 = pd.read_csv('../data/exp/1p36_Expression.csv')  # Expression data
df2 = pd.read_csv('../data/exp/CDK11B_CRISPR_Project_Score_CERES.csv')  # CRISPR scores

# Calculate the average across CDK11A, CDK11B, and CCNL2 columns
df1['average_CDK11_CCNL2'] = df1[['CDK11A', 'CDK11B', 'CCNL2']].mean(axis=1)

# Define percentile thresholds for categorizing gene expression
low_exp_threshold = df1['average_CDK11_CCNL2'].quantile(0.1)
medium_exp_threshold = df1['average_CDK11_CCNL2'].quantile(0.65)

# Categorize gene expression into "Low Expression," "Medium Expression," and "High Expression"
df1['gene_exp_cat'] = pd.cut(df1['average_CDK11_CCNL2'], 
                             bins=[-float('inf'), low_exp_threshold, medium_exp_threshold, float('inf')], 
                             labels=['Low Expression', 'Medium Expression', 'High Expression'])

# Merge the two dataframes on depmap_id
merged_df = pd.merge(df1[['depmap_id', 'gene_exp_cat']], df2, left_on='depmap_id', right_on='Depmap ID')

# Set the context for increased font sizes
sns.set_context("talk", font_scale=7)

# Initialize the figure
plt.figure(figsize=(28, 35), dpi=300)

# Correct the order for violin and scatter plots
correct_order = ['High Expression', 'Medium Expression', 'Low Expression']

# Create violin plot
ax = sns.violinplot(x='gene_exp_cat', y='CRISPR (Project Score, CERES)', data=merged_df,
                    order=correct_order, color="0.8", density_norm="count", inner='point', bw_adjust=1.5)

# Overlay scatter plot
sns.swarmplot(x='gene_exp_cat', y='CRISPR (Project Score, CERES)', data=merged_df,
              order=correct_order, color='black', size=25, ax=ax)

# Calculate and plot the medians
means = merged_df.groupby('gene_exp_cat')['CRISPR (Project Score, CERES)'].mean().reindex(correct_order).values
for i, mean in enumerate(means):
    plt.plot(
        [i - 0.2, i + 0.2],  # Adjust the horizontal span to align with the violin width
        [mean, mean],    # Keep the y-coordinate constant at the median value
        color='red', 
        linestyle='-', 
        linewidth=16, zorder=10
    )
print(means)    
# Customize the x-axis tick labels with the new order

plt.xticks([0,1,2], ['High \nExp.', 'Medium \nExp.', 'Low \nExp.'], 
           fontsize=160, horizontalalignment='center')
ax.tick_params(axis='x', length=0,pad=15) 

ax.tick_params(axis='y', labelsize=120)
for tick in ax.get_yticklabels():
    tick.set_fontweight('bold')

# Customize plot spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(10)   # Left spine (y-axis)
ax.spines['bottom'].set_linewidth(10) # Bottom spine (x-axis)



# Perform Mann-Whitney U statistical tests
high_vs_medium = ttest_ind(
    merged_df[merged_df['gene_exp_cat'] == 'High Expression']['CRISPR (Project Score, CERES)'].dropna(),
    merged_df[merged_df['gene_exp_cat'] == 'Medium Expression']['CRISPR (Project Score, CERES)'].dropna(), alternative='two-sided', equal_var=False
)
medium_vs_low = ttest_ind(
    merged_df[merged_df['gene_exp_cat'] == 'Medium Expression']['CRISPR (Project Score, CERES)'].dropna(),
    merged_df[merged_df['gene_exp_cat'] == 'Low Expression']['CRISPR (Project Score, CERES)'].dropna(), alternative='two-sided', equal_var=False
)
high_vs_low = ttest_ind(
    merged_df[merged_df['gene_exp_cat'] == 'High Expression']['CRISPR (Project Score, CERES)'].dropna(),
    merged_df[merged_df['gene_exp_cat'] == 'Low Expression']['CRISPR (Project Score, CERES)'].dropna(), alternative='two-sided', equal_var=False
)

# Function to add statistical annotation
def add_stat_annotation(ax, p_value, x1, x2, y, h, star_offset=-0.2, color='k'):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=6.5, color=color)
    if p_value < 0.001:
        star_label = '***'
    elif p_value < 0.01:
        star_label = '**'
    elif p_value < 0.05:
        star_label = '*'
    else:
        star_label = 'ns'  # Not significant
        
    if star_label=='ns':
        ax.text(   (x1 + x2) * 0.5, y + h, star_label, ha='center', va='bottom', color=color, fontsize=140)
    else:
        ax.text((x1 + x2) * 0.5, y + h + star_offset, star_label, ha='center', va='bottom', color=color, fontsize=240)

#delta_high_medium = means[0] - means[1]
#delta_medium_low = means[1] - means[2]
#delta_high_low = means[0] - means[2]

# Get the y limits of the plot
y_max = merged_df['CRISPR (Project Score, CERES)'].max()
# Add significance bars, stars, and deltas
add_stat_annotation(plt.gca(), high_vs_medium.pvalue,  0, 1, y_max + 0.4, 0.1)
add_stat_annotation(plt.gca(), medium_vs_low.pvalue,  1, 2, y_max + 0.6, 0.1)
add_stat_annotation(plt.gca(), high_vs_low.pvalue, 0, 2, y_max + 0.8, 0.1)


# Add titles and labels
plt.title('CDK11: Gene Expression \nScreen', fontsize=180, pad=90, fontweight='bold')
plt.xlabel('Gene Expression', fontsize=180, labelpad=60, fontweight='bold')
plt.ylabel('CDK11B Dependency \nScore', fontsize=180, fontweight='bold')

# Save the plot
# plt.savefig('../Gene_Exp_Dep_Score_QL.svg', bbox_inches='tight', dpi=300, pad_inches=0.5)
plt.show()


In [ ]:
print(wild_vs_shallow.pvalue)
print(shallow_vs_deep.pvalue)
print(wild_vs_deep.pvalue)

# 4E

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load the CSV files
df1 = pd.read_csv('../data/rnai/1p36_CN.csv')  # Adjusted file path
df2 = pd.read_csv('../data/rnai/CDK11B_RNAi_Achilles_DEMETER2.csv')

# Calculate the average across CDK11A, CDK11B, and CCNL2
df1['average_CDK11_CCNL2'] = df1[['CDK11A', 'CDK11B', 'CCNL2']].mean(axis=1)

# Calculate the percentile thresholds for the average values
deep_del_threshold = df1['average_CDK11_CCNL2'].quantile(0.1)
shallow_del_threshold = df1['average_CDK11_CCNL2'].quantile(0.4)

# Segregate df1 into wild type, deep del, and shallow del based on the calculated thresholds
df1['CN_category'] = pd.cut(df1['average_CDK11_CCNL2'], 
                            bins=[-float('inf'), deep_del_threshold, shallow_del_threshold, float('inf')], 
                            labels=['Deep Deletion', 'Shallow Deletion', 'Copy Neutral / Gain'])
merged_df = pd.merge(df1[['depmap_id', 'CN_category']], df2, left_on='depmap_id', right_on='Depmap ID')
sns.set_context("talk", font_scale=7)

# Initialize the figure
plt.figure(figsize=(32, 35), dpi=300)

custom_order = ['Copy Neutral / Gain', 'Shallow Deletion', 'Deep Deletion']
x_positions = {cat: pos for cat, pos in zip(custom_order, [0,1,2])}
#x_pos=[0,1.1,1.75]
# Prepare the data
plot_data = merged_df[merged_df['CN_category'].isin(custom_order)]
plot_data['x_pos'] = plot_data['CN_category'].map(x_positions)
plot_data['x_pos'] = pd.to_numeric(plot_data['x_pos'])
plot_data = plot_data.sort_values('x_pos', ascending=True)

# Create violin plot
ax = sns.violinplot(x='x_pos', y='RNAi (Achilles, DEMETER2)', data=plot_data,
                    color=".8", density_norm="count", inner='point', bw_adjust=1.5)

# Overlay scatter plot
sns.swarmplot(x='x_pos', y='RNAi (Achilles, DEMETER2)', data=plot_data,
              color='black', size=20, ax=ax)

# Set x-ticks and labels
plt.xticks([0,1,2], ['Copy \n Neutral \n/ Gain', 'Shallow \nDeletion', 'Deep \nDeletion'], 
           fontsize=140, horizontalalignment='center')
ax.tick_params(axis='x', length=0,pad=15) 
# Customize the y-axis tick labels
ax.tick_params(axis='y', labelsize=120)
for tick in ax.get_yticklabels():
    tick.set_fontweight('bold')

# Customize plot spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(10)   # Left spine (y-axis)
ax.spines['bottom'].set_linewidth(10) # Bottom spine (x-axis)

# Calculate and plot the medians
means = merged_df.groupby('CN_category')['RNAi (Achilles, DEMETER2)'].mean().reindex(
    ['Copy Neutral / Gain', 'Shallow Deletion', 'Deep Deletion']
).values
for i, mean in enumerate(means):
    plt.plot(
        [i - 0.2, i + 0.2],  # Adjust the horizontal span to align with the violin width
        [mean, mean],    # Keep the y-coordinate constant at the median value
        color='red', 
        linestyle='-', 
        linewidth=12, zorder=10
    )

# Perform Mann-Whitney U statistical tests
wild_vs_shallow = ttest_ind(
    merged_df[merged_df['CN_category'] == 'Copy Neutral / Gain']['RNAi (Achilles, DEMETER2)'].dropna(),
    merged_df[merged_df['CN_category'] == 'Shallow Deletion']['RNAi (Achilles, DEMETER2)'].dropna(), alternative='two-sided', equal_var=False
)
shallow_vs_deep = ttest_ind(
    merged_df[merged_df['CN_category'] == 'Shallow Deletion']['RNAi (Achilles, DEMETER2)'].dropna(),
    merged_df[merged_df['CN_category'] == 'Deep Deletion']['RNAi (Achilles, DEMETER2)'].dropna(), alternative='two-sided', equal_var=False
)
wild_vs_deep = ttest_ind(
    merged_df[merged_df['CN_category'] == 'Copy Neutral / Gain']['RNAi (Achilles, DEMETER2)'].dropna(),
    merged_df[merged_df['CN_category'] == 'Deep Deletion']['RNAi (Achilles, DEMETER2)'].dropna(), alternative='two-sided', equal_var=False
)


# Get the y limits of the plot
y_max = merged_df['RNAi (Achilles, DEMETER2)'].max()

#delta_high_medium = means[0] - means[1]
#delta_medium_low = means[1] - means[2]
#delta_high_low = means[0] - means[2]

def add_stat_annotation(ax, p_value, x1, x2, y, h, star_offset=-0.2, delta_offset=-0.1, color='k'):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=6.5, color=color)
    if p_value < 0.001:
        star_label = '***'
    elif p_value < 0.01:
        star_label = '**'
    elif p_value < 0.05:
        star_label = '*'
    else:
        star_label = 'ns'
    ax.text((x1 + x2) * 0.5, y + h + star_offset, star_label, ha='center', va='bottom', color=color, fontsize=240)
    #ax.text((x1 + x2) * 0.5, y + 0.5*h + delta_offset, f"Δ={delta:.2f}", ha='center', va='bottom', color='black', fontsize=90)

# Add significance bars, stars, and deltas
add_stat_annotation(plt.gca(), wild_vs_shallow.pvalue,  0, 1, y_max + 0.4, 0.1)
add_stat_annotation(plt.gca(), shallow_vs_deep.pvalue,  1, 2, y_max + 0.7, 0.1)
add_stat_annotation(plt.gca(), wild_vs_deep.pvalue, 0, 2, y_max + 1, 0.1)

plt.title('CDK11: RNAi Dependency', fontsize=180,pad=90, fontweight='bold')
plt.xlabel('Chr. 1p36 Copy Number', fontsize=180, labelpad=60, fontweight='bold')
plt.ylabel('CDK11B Dependency \nScore', fontsize=180, fontweight='bold')
# plt.savefig('../CDK11B_RNAi_QL.svg',bbox_inches='tight',dpi=300,pad_inches=0.5)
plt.show()


In [ ]:
print(wild_vs_shallow.pvalue)
print(shallow_vs_deep.pvalue)
print(wild_vs_deep.pvalue)

# 4F

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load the CSV files
df1 = pd.read_csv('../data/crispr/1p36_CN.csv')  # CN data
df2 = pd.read_csv('../data/drug_sensitivity/MTS023_JASON_SHELTZER_LFC_MATRIX_OTS964.csv')
df2['Cell Line Name'] = df2['Cell Line Name'].apply(lambda x: x.split('_')[0] if isinstance(x, str) else x)

# Define the concentration column to analyze
conc = 'OTS964_0.1235 uM'

# Calculate the average across CDK11A, CDK11B, and CCNL2 in df1
df1['average_CDK11_CCNL2'] = df1[['CDK11A', 'CDK11B', 'CCNL2']].mean(axis=1)

# Calculate the percentile thresholds for the average values
deep_del_threshold = df1['average_CDK11_CCNL2'].quantile(0.1)
shallow_del_threshold = df1['average_CDK11_CCNL2'].quantile(0.4)

# Categorize copy number into "Deep Deletion," "Shallow Deletion," and "Copy Neutral / Gain"
df1['CN_category'] = pd.cut(df1['average_CDK11_CCNL2'], 
                            bins=[-float('inf'), deep_del_threshold, shallow_del_threshold, float('inf')], 
                            labels=['Deep Deletion', 'Shallow Deletion', 'Copy Neutral / Gain'])

# Merge df1 and df2 on the cell line display name
merged_df = pd.merge(df1[['depmap_id', 'CN_category', 'cell_line_display_name']], df2, 
                     left_on='cell_line_display_name', right_on='Cell Line Name')

# Set the context for increased font sizes
sns.set_context("talk", font_scale=7)

# Initialize the figure
plt.figure(figsize=(32,35), dpi=300)


# Create violin plot
ax = sns.violinplot(x='CN_category', y=conc, data=merged_df,
                    order=['Copy Neutral / Gain', 'Shallow Deletion', 'Deep Deletion'], 
                    color=".8", density_norm="count", inner='point', bw_adjust=1.5)

# Overlay scatter plot
sns.swarmplot(x='CN_category', y=conc, data=merged_df,
              order=['Copy Neutral / Gain', 'Shallow Deletion', 'Deep Deletion'], 
              color='black', size=15, ax=ax)

# Calculate and plot the means
category_order = ['Copy Neutral / Gain', 'Shallow Deletion', 'Deep Deletion']

# Get the means using the same category order, drop NaNs safely
means = merged_df.groupby('CN_category')[conc].mean().reindex(category_order)

# Print means to confirm
print(means)

# Plot red mean lines only for the categories that exist
for i, (cat, mean) in enumerate(means.items()):
    if pd.notna(mean):
        plt.plot(
            [i - 0.2, i + 0.2],
            [mean, mean],
            color='red',
            linestyle='-',
            linewidth=16,
            zorder=10
        )

# Add significance annotations
#delta_high_medium = means[0] - means[1]
#delta_medium_low = means[1] - means[2]
#delta_high_low = means[0] - means[2]

# Customize the x-axis tick labels with line breaks
plt.xticks([0,1,2], ['Copy \n Neutral \n/ Gain', 'Shallow \nDeletion', 'Deep \nDeletion'], 
           fontsize=140, horizontalalignment='center')
ax.tick_params(axis='x', length=0,pad=15) 

# Customize the y-axis tick labels
ax.tick_params(axis='y', labelsize=120)
for tick in ax.get_yticklabels():
    tick.set_fontweight('bold')
    
# Customize plot spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(10)   # Left spine (y-axis)
ax.spines['bottom'].set_linewidth(10) # Bottom spine (x-axis)


# Perform t-tests
wild_vs_shallow = ttest_ind(
    merged_df[merged_df['CN_category'] == 'Copy Neutral / Gain'][conc].dropna(),
    merged_df[merged_df['CN_category'] == 'Shallow Deletion'][conc].dropna(), alternative='two-sided', equal_var=False
)
shallow_vs_deep = ttest_ind(
    merged_df[merged_df['CN_category'] == 'Shallow Deletion'][conc].dropna(),
    merged_df[merged_df['CN_category'] == 'Deep Deletion'][conc].dropna(), alternative='two-sided', equal_var=False
)

wild_vs_deep = ttest_ind(
    merged_df[merged_df['CN_category'] == 'Copy Neutral / Gain'][conc].dropna(),
    merged_df[merged_df['CN_category'] == 'Deep Deletion'][conc].dropna(), alternative='two-sided', equal_var=False

)

def add_stat_annotation(ax, p_value, x1, x2, y, h, star_offset=-0.6, delta_offset=-0.7, color='k'):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=6.5, color=color)
    if p_value < 0.001:
        star_label = '***'
    elif p_value < 0.01:
        star_label = '**'
    elif p_value < 0.05:
        star_label = '*'
    else:
        star_label = 'ns'  # Not significant
        
    if star_label=='ns':
        ax.text((x1 + x2) * 0.5, y + h, star_label, ha='center', va='bottom', color=color, fontsize=140)
    else:
        ax.text((x1 + x2) * 0.5, y + h + star_offset, star_label, ha='center', va='bottom', color=color, fontsize=240)

    #ax.text((x1 + x2) * 0.5, y + h + delta_offset, f"Δ={delta:.2f}", ha='center', va='bottom', color='black', fontsize=90)

y_max = merged_df[conc].max()
# Add significance bars, stars, and deltas
add_stat_annotation(plt.gca(), wild_vs_shallow.pvalue, 0, 1, y_max + 1, 0.5)
add_stat_annotation(plt.gca(), shallow_vs_deep.pvalue,  1, 2, y_max + 2, 0.5)
add_stat_annotation(plt.gca(), wild_vs_deep.pvalue,  0, 2, y_max + 3, 0.5)

# Add titles and labels
plt.title('OTS964 Cancer Cell Line \nSensitivity', fontsize=180, pad=90, fontweight='bold')
plt.xlabel('Chr. 1p36 Copy Number', fontsize=180, labelpad=60, fontweight='bold')
plt.ylabel('OTS964 Log-Fold \nChange', fontsize=180, fontweight='bold')

# Save the plot
# plt.savefig('../OTS964_QL.svg', bbox_inches='tight', dpi=300, pad_inches=0.5)
plt.show()


In [ ]:
print(wild_vs_shallow.pvalue)
print(shallow_vs_deep.pvalue)
print(wild_vs_deep.pvalue)